# Business Entity Resolution - Kaggle runner

Inputs (add both with **+ Add Input**):
1. the challenge zip uploaded as a private Kaggle Dataset (Kaggle unzips it),
2. this `business_entity_resolution` folder uploaded as a second private Dataset.

Settings: **Internet ON** (only to `pip install` three small libraries), accelerator **None** or **TPU VM**
(the TPU VM gives many more CPU cores; nothing here uses a GPU). No data leaves the notebook.


In [ ]:
import glob, os, subprocess, sys
def find(pattern):
    hits = [p for p in glob.glob('/kaggle/input/**/' + pattern, recursive=True) if '__MACOSX' not in p]
    assert hits, pattern
    return hits[0]
DATA = os.path.dirname(os.path.dirname(find('train/train_source1.tsv')))   # .../dataset
CODE_SRC = os.path.dirname(os.path.dirname(find('src/pipeline.py')))        # .../business_entity_resolution
VALIDATOR = find('utils/validate_submission.py')
print(DATA, CODE_SRC, VALIDATOR, os.cpu_count(), 'CPUs')

In [ ]:
!rm -rf /kaggle/working/ber && cp -r "$CODE_SRC" /kaggle/working/ber
!pip install -q rapidfuzz==3.14.5 sparse_dot_topn==1.2.0 Unidecode==1.4.0
%cd /kaggle/working/ber

In [ ]:
# ~train_frac of the training S1 entities (by whole states) are used; 20% of those are held out
# to report validation F0.5 and tune the decision threshold.
!python -u -m src.pipeline train --data-dir "$DATA" --work /kaggle/working/artifacts 2>&1 | grep -v Warning

In [ ]:
!python -u -m src.pipeline predict --data-dir "$DATA" --work /kaggle/working/artifacts --out /kaggle/working/output

In [ ]:
!python "$VALIDATOR" --matching /kaggle/working/output/matching_results.tsv --candidate /kaggle/working/output/candidate_pairs.tsv --test-dir "$DATA/test" 

In [ ]:
!cat /kaggle/working/artifacts/meta.json | head -5
!ls -la /kaggle/working/output